In [1]:
# Mount Drive & Load Both Models

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import tensorflow as tf
import pickle
import numpy as np

MODEL_PATH = '/content/drive/MyDrive/FaceAttendance/models'
DATA_PATH  = '/content/drive/MyDrive/FaceAttendance/data'

# Load models
face_model  = tf.keras.models.load_model(f'{MODEL_PATH}/face_recognition.h5')
attn_model  = tf.keras.models.load_model(f'{MODEL_PATH}/attention_classifier.h5')

# Load label encoder
with open(f'{DATA_PATH}/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

print('Both models loaded!')
print(f'   Known people: {list(le.classes_)}')

Mounted at /content/drive


Both models loaded!
   Known people: [np.str_('Alejandro_Toledo'), np.str_('Alvaro_Uribe'), np.str_('Amelie_Mauresmo'), np.str_('Andre_Agassi'), np.str_('Angelina_Jolie'), np.str_('Ariel_Sharon'), np.str_('Arnold_Schwarzenegger'), np.str_('Atal_Bihari_Vajpayee'), np.str_('Bill_Clinton'), np.str_('Carlos_Menem'), np.str_('Colin_Powell'), np.str_('David_Beckham'), np.str_('Donald_Rumsfeld'), np.str_('George_Robertson'), np.str_('George_W_Bush'), np.str_('Gerhard_Schroeder'), np.str_('Gloria_Macapagal_Arroyo'), np.str_('Gray_Davis'), np.str_('Guillermo_Coria'), np.str_('Hamid_Karzai'), np.str_('Hans_Blix'), np.str_('Hugo_Chavez'), np.str_('Igor_Ivanov'), np.str_('Jack_Straw'), np.str_('Jacques_Chirac'), np.str_('Jean_Chretien'), np.str_('Jennifer_Aniston'), np.str_('Jennifer_Capriati'), np.str_('Jennifer_Lopez'), np.str_('Jeremy_Greenstock'), np.str_('Jiang_Zemin'), np.str_('John_Ashcroft'), np.str_('John_Negroponte'), np.str_('Jose_Maria_Aznar'), np.str_('Juan_Carlos_Ferrero'), np.str_('

In [2]:
# Install & Setup

!pip install mtcnn -q
!pip install lz4 -q
!pip install python-lz4 -q
!pip install --upgrade joblib -q

# Ensure lz4 is explicitly loaded and then re-import joblib to pick up lz4
import lz4 # Explicitly import lz4 to ensure it's loaded
import joblib # Import joblib
import importlib
importlib.reload(joblib) # This might help joblib to re-evaluate its environment

from mtcnn import MTCNN
import cv2, pandas as pd, os
from datetime import datetime

detector = MTCNN()

# Output CSV path
OUTPUT_PATH = '/content/drive/MyDrive/FaceAttendance/reports'
os.makedirs(OUTPUT_PATH, exist_ok=True)

print('Ready!')

ERROR: Could not find a version that satisfies the requirement python-lz4 (from versions: none)
ERROR: No matching distribution found for python-lz4
Ready!


In [3]:

# Helper Functions


def predict_face(face_img):
    """Predict who the person is"""
    img = cv2.resize(face_img, (224, 224)) / 255.0
    img = np.expand_dims(img, axis=0)
    preds = face_model.predict(img, verbose=0)
    confidence = np.max(preds)
    label = le.inverse_transform([np.argmax(preds)])[0]
    return label, confidence

def predict_attention(face_img):
    """Predict attention level (0-100%)"""
    img = cv2.resize(face_img, (64, 64)) / 255.0
    img = np.expand_dims(img, axis=0)
    score = attn_model.predict(img, verbose=0)[0][0]
    return round(float(score) * 100, 1)  # returns 0-100%

def log_to_csv(records, filename):
    """Save records to CSV"""
    df = pd.DataFrame(records)
    df.to_csv(filename, index=False)
    print(f'Saved: {filename}')

print('Helper functions ready!')

Helper functions ready!


In [4]:
# CELL 4 — Live Preview with Detection Overlay + Auto Capture after 4 seconds
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2, numpy as np
from datetime import datetime

def take_photo_with_preview(filename='photo.jpg'):
    js = Javascript('''
    async function takePhotoWithPreview() {
        const div = document.createElement('div');
        div.style.textAlign = 'center';
        document.body.appendChild(div);

        // Video element
        const video = document.createElement('video');
        video.style.width = '400px';
        video.style.borderRadius = '10px';
        video.style.display = 'block';
        video.style.margin = '0 auto';
        video.autoplay = true;
        div.appendChild(video);

        // Canvas overlay for drawing boxes
        const canvas = document.createElement('canvas');
        canvas.style.position = 'absolute';
        canvas.style.top = '0';
        canvas.style.left = '0';
        div.style.position = 'relative';
        div.style.display = 'inline-block';
        div.appendChild(canvas);

        // Countdown text
        const countdown = document.createElement('div');
        countdown.style.fontSize = '24px';
        countdown.style.fontWeight = 'bold';
        countdown.style.marginTop = '10px';
        countdown.style.color = '#00f5d4';
        div.appendChild(countdown);

        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;

        await new Promise(r => video.onloadedmetadata = r);
        canvas.width  = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.style.width  = '400px';
        canvas.style.height = (video.videoHeight * 400 / video.videoWidth) + 'px';

        // Wait before countdown
        await new Promise(r => setTimeout(r, 1000));

        // Countdown 3..2..1
        for (let i = 3; i > 0; i--) {
            countdown.textContent = 'Capturing in ' + i + ' seconds...';
            await new Promise(r => setTimeout(r, 1000));
        }
        countdown.textContent = 'Captured!';
        countdown.style.color = '#10b981';

        // Capture final frame
        const captureCanvas = document.createElement('canvas');
        captureCanvas.width  = video.videoWidth;
        captureCanvas.height = video.videoHeight;
        captureCanvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(t => t.stop());
        await new Promise(r => setTimeout(r, 600));
        div.remove();

        return captureCanvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data   = eval_js('takePhotoWithPreview()')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print('Ready - live preview with 3 second countdown!')

Ready - live preview with 3 second countdown!


In [8]:
# CELL 5 — RUN SESSION
import cv2
from datetime import datetime
from IPython.display import Image as IPImage, display as ipy_display

CONFIDENCE_THRESHOLD = 0.6
NUM_FRAMES = 5
records = []

print(f'Starting session - {NUM_FRAMES} frames')
print('Live preview shown, auto captures after 4 seconds\n')

for i in range(NUM_FRAMES):
    print(f'--- Frame {i+1}/{NUM_FRAMES} ---')

    filename = take_photo_with_preview(f'/content/frame_{i}.jpg')

    img     = cv2.imread(filename)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces   = detector.detect_faces(img_rgb)
    now     = datetime.now()

    if not faces:
        print('  No face detected\n')
        continue

    for face_data in faces:
        x, y, w, h = face_data['box']
        x, y       = max(0, x), max(0, y)
        face_crop  = img_rgb[y:y+h, x:x+w]

        name, confidence = predict_face(face_crop)
        attention_pct    = predict_attention(face_crop)
        status           = 'Attentive' if attention_pct >= 50 else 'Distracted'

        # Text result only
        print(f'  Name       : {name}')
        print(f'  Confidence : {confidence*100:.1f}%')
        print(f'  Attention  : {attention_pct}%')
        print(f'  Status     : {status}')
        print()

        if confidence >= CONFIDENCE_THRESHOLD:
            records.append({
                'Name':        name,
                'Date':        now.strftime('%Y-%m-%d'),
                'Time':        now.strftime('%H:%M:%S'),
                'Present':     'Yes',
                'Confidence':  f'{confidence*100:.1f}%',
                'Attention_%': attention_pct,
                'Status':      status
            })

print('Session complete!')

Starting session - 5 frames
Live preview shown, auto captures after 4 seconds

--- Frame 1/5 ---


<IPython.core.display.Javascript object>

  Name       : jianna
  Confidence : 40.9%
  Attention  : 76.6%
  Status     : Attentive

--- Frame 2/5 ---


<IPython.core.display.Javascript object>

  Name       : jianna
  Confidence : 63.0%
  Attention  : 88.1%
  Status     : Attentive

--- Frame 3/5 ---


...

  Name       : sreelakshmi
  Confidence : 53.0%
  Attention  : 73.1%
  Status     : Attentive

--- Frame 4/5 ---


<IPython.core.display.Javascript object>

  Name       : anushka
  Confidence : 64.9%
  Attention  : 5.4%
  Status     : Distracted

--- Frame 5/5 ---


<IPython.core.display.Javascript object>

  Name       : anushka
  Confidence : 47.0%
  Attention  : 3.5%
  Status     : Distracted

Session complete!


In [ ]:
# CELL 6 — Generate & Append CSV Report
import pandas as pd, os
from datetime import datetime

OUTPUT_PATH = '/content/drive/MyDrive/FaceAttendance/reports'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Fixed filenames — same file gets appended every run
RAW_FILE  = f'{OUTPUT_PATH}/raw_log_ongoing.csv'
SUMM_FILE = f'{OUTPUT_PATH}/attendance_report_ongoing.csv'

if records:
    df = pd.DataFrame(records)

    # Append to raw log
    if os.path.exists(RAW_FILE):
        existing = pd.read_csv(RAW_FILE)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(RAW_FILE, index=False)

    # Summary (recalculate from ALL data including previous runs)
    summary = df.groupby('Name').agg(
        Date              = ('Date',        'first'),
        First_Seen        = ('Time',        'min'),
        Last_Seen         = ('Time',        'max'),
        Present           = ('Present',     'first'),
        Avg_Attention_pct = ('Attention_%', 'mean')
    ).reset_index()

    summary['Avg_Attention_pct'] = summary['Avg_Attention_pct'].round(1)
    summary['Status'] = summary['Avg_Attention_pct'].apply(
        lambda x: 'Attentive' if x >= 50 else 'Distracted')

    summary.to_csv(SUMM_FILE, index=False)

    print('\nATTENDANCE + ATTENTION REPORT (All runs combined)')
    print('-'*60)
    print(summary.to_string(index=False))
    print('-'*60)
    print(f'\nRaw log:  {RAW_FILE}')
    print(f'Summary:  {SUMM_FILE}')

    # ── Option to reset ──
    reset = input('\nType RESET to clear all data, or press Enter to keep: ')
    if reset.strip().upper() == 'RESET':
        os.remove(RAW_FILE)
        os.remove(SUMM_FILE)
        print('All data cleared.')
    else:
        print('Data added')

else:
    print('No records to save!')


ATTENDANCE + ATTENTION REPORT (All runs combined)
------------------------------------------------------------
   Name       Date First_Seen Last_Seen Present  Avg_Attention_pct     Status
anushka 2026-04-13   08:06:54  08:06:54     Yes                5.4 Distracted
 jianna 2026-03-30   08:06:40  08:12:57     Yes               84.2  Attentive
------------------------------------------------------------

Raw log:  /content/drive/MyDrive/FaceAttendance/reports/raw_log_ongoing.csv
Summary:  /content/drive/MyDrive/FaceAttendance/reports/attendance_report_ongoing.csv
